# 5 Statistical Tests

Wilcoxon signed-rank tests, energy analysis, learned-beta analysis, and the publication tables.

Paths are imported from `config.py` at the repository root.
Run the notebooks in numbered order; see the repository README.

# Post-Experiment Statistical Significance Tests

Run this **after** all 10 encoders have been trained by `4_0_Experiment_of_encoders_for_1D_SNN.ipynb`
and the result CSVs have been saved to `RESULTS_DIR`.

This notebook produces:

1. **Statistical significance tests** — Friedman + pairwise Wilcoxon signed-rank (paired by fold/seed)
2. **Synaptic operations & energy** — read directly from measured test-set values (no estimates)
3. **Learned β analysis** — parsed from the `betas` column of the CV CSV
4. **DASEventNet comparison** — your SNN vs the published CNN baseline
5. **Publication-ready summary tables** — CV, test set, sweep parameters

## Encoders

The 10-encoder benchmark from `4_0_Experiment_of_encoders_for_1D_SNN.ipynb`:

| Type     | Encoders                                                    |
|----------|-------------------------------------------------------------|
| Baseline | Rate, Phase, TTFS                                            |
| Proposed | Delta-Mod, PDE, ATDE, MASTE, ST-MW, AMSTE, SFBE              |

All encoder parameters were chosen via the multi-file sweep in
`2_2_FULL_Parameter_Sensitivity_Sweep__Multiple_data_.ipynb` (400 event + 400 noise files,
sparsity target 0.3–1.5%) and applied to the full 2k+2k batch in
`3_0_Preprocessing__2k_files_.ipynb`.

## Inputs

- `RESULTS_DIR/mode1_within_cv_results.csv` — 5 folds × 3 seeds = 15 rows per encoder
- `RESULTS_DIR/mode1_test_results.csv` — 1 row per encoder (best fold/seed evaluated on the held-out test set, with measured efficiency metrics)


In [ ]:
# --- repository configuration -------------------------------------
# All paths come from config.py at the repository root. Edit that
# file once; do not hardcode paths in this notebook.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import config
# ------------------------------------------------------------------

# =============================================================================
# CELL 1 — Imports & Configuration
# =============================================================================
import os
import ast
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, friedmanchisquare
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# ── Paths (must match 4_0_Experiment_of_encoders_for_1D_SNN.ipynb) ───────────
RESULTS_DIR = config.RESULTS_DIR
CV_CSV      = os.path.join(RESULTS_DIR, 'mode1_within_cv_results.csv')
TEST_CSV    = os.path.join(RESULTS_DIR, 'mode1_test_results.csv')

# ── Encoders (must match 4_0_Experiment_of_encoders_for_1D_SNN.ipynb) ────────
ENCODER_ORDER = [
    'Rate', 'Phase', 'TTFS',
    'Delta-Mod', 'PDE', 'ATDE', 'MASTE',
    'ST-MW', 'AMSTE', 'SFBE',
]

ENCODER_CATEGORY = {
    'Rate':      'Baseline', 'Phase':     'Baseline', 'TTFS':  'Baseline',
    'Delta-Mod': 'Proposed', 'PDE':       'Proposed', 'ATDE':  'Proposed',
    'MASTE':     'Proposed', 'ST-MW':     'Proposed', 'AMSTE': 'Proposed',
    'SFBE':      'Proposed',
}

# ── Metrics saved by nb4 ──────────────────────────────────────────────────────
# Primary classification metrics — what the paper reports first
PRIMARY_METRICS   = ['f1', 'recall', 'fnr', 'fpr', 'auprc']
# Secondary classification metrics — supporting evidence
SECONDARY_METRICS = ['accuracy', 'balanced_acc', 'precision', 'specificity', 'auc', 'mcc']
# Efficiency metrics — only on test CSV, computed by compute_efficiency_metrics()
EFFICIENCY_METRICS = ['input_spike_rate', 'l1_spike_rate', 'l2_spike_rate',
                      'synops', 'msynops', 'energy_uj', 'inference_time_ms']

# ── SNN architecture (must match nb4) ────────────────────────────────────────
NUM_INPUTS  = 366
NUM_HIDDEN1 = 128
NUM_HIDDEN2 = 64
NUM_OUTPUTS = 2
NUM_STEPS   = 500

# Loihi-2 energy figure: 23.6 pJ **per synaptic operation** (NOT per spike).
# This is the figure used by compute_efficiency_metrics() in nb4, so the
# energy_uj column in the test CSV is already correct — we only redefine
# the constant here for the DASEventNet comparison below.
ENERGY_PER_SYNOP_PJ = 23.6

# DASEventNet (ResNet-50) baseline — Yu et al., JGR Solid Earth 2024
RESNET50_PARAMS              = 25_600_000
RESNET50_FLOPS               = 3_800_000_000
RESNET50_ENERGY_PJ_PER_FLOP  = 7.0  # GPU FP32 estimate

# ── Experiment design ────────────────────────────────────────────────────────
N_TOTAL_FILES = 2000
TEST_FRAC     = 0.15
N_TEST        = int(round(N_TOTAL_FILES * TEST_FRAC))    # 600
N_TRAIN_VAL   = N_TOTAL_FILES - N_TEST                    # 3,400
N_FOLDS       = 5
N_SEEDS       = 3
N_RUNS        = N_FOLDS * N_SEEDS                         # 15

print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"Encoders    : {ENCODER_ORDER}")
print(f"Test set    : {N_TEST} files | Train/Val pool: {N_TRAIN_VAL} | "
      f"CV runs/encoder: {N_RUNS} ({N_FOLDS} folds × {N_SEEDS} seeds)")


## Section 1 — Load Results

Loads both CSVs, builds an `(encoder, run_id) → metrics` table for paired
statistical tests, and parses the stringified `betas` dict written by nb4
(`{'b1': ..., 'b2': ..., 'b3': ...}`) into separate `beta1`, `beta2`, `beta3`
columns.

In [ ]:
# =============================================================================
# CELL 2 — Load CV + Test CSVs
# =============================================================================
print("=" * 80)
print("LOADING RESULTS")
print("=" * 80)

# ── CV results ────────────────────────────────────────────────────────────────
if not os.path.exists(CV_CSV):
    raise FileNotFoundError(f"CV CSV not found: {CV_CSV}\n"
                            f"Run mode1_all() in nb4 first.")
cv_df = pd.read_csv(CV_CSV)
print(f"  CV rows         : {len(cv_df)}")
print(f"  CV columns      : {list(cv_df.columns)}")

# ── Test results ──────────────────────────────────────────────────────────────
if not os.path.exists(TEST_CSV):
    raise FileNotFoundError(f"Test CSV not found: {TEST_CSV}\n"
                            f"Run mode1_all(run_test_eval=True) in nb4 first.")
test_df = pd.read_csv(TEST_CSV)
print(f"  Test rows       : {len(test_df)}")
print(f"  Test columns    : {list(test_df.columns)}")

# ── Parse betas dict column ───────────────────────────────────────────────────
# nb4 stores betas as a stringified dict: "{'b1': 0.85, 'b2': 0.90, 'b3': 0.95}"
def parse_betas(s):
    if pd.isna(s):
        return {'b1': np.nan, 'b2': np.nan, 'b3': np.nan}
    if isinstance(s, dict):
        return s
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return {'b1': np.nan, 'b2': np.nan, 'b3': np.nan}

if 'betas' in cv_df.columns:
    parsed = cv_df['betas'].apply(parse_betas)
    cv_df['beta1'] = parsed.apply(lambda d: d.get('b1', np.nan))
    cv_df['beta2'] = parsed.apply(lambda d: d.get('b2', np.nan))
    cv_df['beta3'] = parsed.apply(lambda d: d.get('b3', np.nan))
    print(f"  Parsed `betas`  : beta1, beta2, beta3 columns added")
else:
    print("  WARNING: `betas` column missing — beta analysis will be skipped")
    cv_df['beta1'] = np.nan
    cv_df['beta2'] = np.nan
    cv_df['beta3'] = np.nan

# ── Verify all encoders present + report run counts ──────────────────────────
print(f"\n  {'Encoder':>12s}  {'Category':>9s}  {'CV runs':>8s}  {'Test rows':>10s}  {'Status':>10s}")
print("-" * 65)
active_encoders = []
for enc in ENCODER_ORDER:
    n_cv   = (cv_df['encoder']   == enc).sum()
    n_test = (test_df['encoder'] == enc).sum()
    status = 'OK' if (n_cv >= 2 and n_test >= 1) else 'MISSING'
    if n_cv >= 2:
        active_encoders.append(enc)
    print(f"  {enc:>12s}  {ENCODER_CATEGORY[enc]:>9s}  {n_cv:>8d}  {n_test:>10d}  {status:>10s}")

print(f"\n  Active encoders for tests: {len(active_encoders)} / {len(ENCODER_ORDER)}")
if len(active_encoders) < 2:
    raise RuntimeError("Need at least 2 encoders with CV results for significance tests.")

# ── Sort each encoder's CV rows by (fold, seed) so runs pair across encoders ─
# This is REQUIRED for paired tests (Wilcoxon, Friedman) to be valid.
cv_df = cv_df.sort_values(['encoder', 'fold', 'seed']).reset_index(drop=True)


## Section 2 — Statistical Significance Tests

**Paired tests.** Each fold/seed combination is a "subject"; each encoder is a
"condition". Because the same fold/seed pair is used for every encoder in nb4,
the runs are paired across encoders and we can use Friedman and Wilcoxon
signed-rank tests.

**Family-wise error.** The pairwise Wilcoxon block reports raw p-values plus
a Bonferroni-corrected significance flag (α / number of pairs) so the reader
can see both the unadjusted result and the conservative correction.

In [ ]:
# =============================================================================
# CELL 3 — Friedman test on F1
# =============================================================================
print("=" * 80)
print("SECTION 1: STATISTICAL SIGNIFICANCE TESTS")
print("=" * 80)

print("\n--- Friedman Test (does encoder choice matter?) ---")

# Build paired arrays: rows = runs (sorted by fold,seed), columns = encoders
def get_paired_metric(df, encoders, metric):
    """Return list of arrays, one per encoder, paired by (fold, seed) index.
    Encoders that don't have all run/seed pairs are skipped."""
    out = []
    keep = []
    for enc in encoders:
        sub = df[df['encoder'] == enc].sort_values(['fold', 'seed'])
        if len(sub) == 0 or metric not in sub.columns:
            continue
        out.append(sub[metric].values)
        keep.append(enc)
    return out, keep

f1_arrays, f1_encoders = get_paired_metric(cv_df, active_encoders, 'f1')
lengths = [len(a) for a in f1_arrays]

if len(set(lengths)) == 1 and lengths and lengths[0] > 0:
    stat, p_friedman = friedmanchisquare(*f1_arrays)
    sig_label = ('*** SIGNIFICANT (p < 0.001)' if p_friedman < 0.001
                 else '** SIGNIFICANT (p < 0.01)' if p_friedman < 0.01
                 else '* SIGNIFICANT (p < 0.05)'  if p_friedman < 0.05
                 else 'NOT significant')
    print(f"  Friedman chi-squared = {stat:.4f},  p-value = {p_friedman:.6e}")
    print(f"  {sig_label}")
    print(f"  Interpretation: "
          f"{'Encoder choice significantly affects F1.' if p_friedman < 0.05 else 'No significant difference between encoders.'}")
    print(f"  Encoders tested: {len(f1_encoders)} | Runs per encoder: {lengths[0]}")
else:
    print(f"  SKIPPED: unequal run counts {dict(zip(f1_encoders, lengths))}")
    p_friedman = None


In [ ]:
# =============================================================================
# CELL 4 — Pairwise Wilcoxon signed-rank tests on F1
# =============================================================================
print(f"\n--- Pairwise Wilcoxon Signed-Rank Tests (F1, paired by fold/seed) ---")

# Build dict of paired arrays for fast lookup
f1_paired = dict(zip(f1_encoders, f1_arrays))
n_pairs = len(list(combinations(f1_encoders, 2)))
bonferroni_alpha = 0.05 / n_pairs if n_pairs > 0 else 0.05
print(f"  Total pairs: {n_pairs} | Bonferroni-corrected α = {bonferroni_alpha:.5f}")
print()
print(f"  {'Encoder A':>12s} vs {'Encoder B':>12s}  | "
      f"{'p-value':>11s}  {'Sig':>4s}  {'Bonf':>5s}  "
      f"{'Winner':>12s}  {'ΔF1':>8s}")
print("-" * 85)

significance_results = []
for enc_a, enc_b in combinations(f1_encoders, 2):
    f1_a = f1_paired[enc_a]
    f1_b = f1_paired[enc_b]

    if len(f1_a) != len(f1_b):
        # Should not happen because we filtered to same length above, but guard anyway
        continue

    diff = f1_a - f1_b
    if np.all(diff == 0):
        p_val = 1.0
    else:
        try:
            _, p_val = wilcoxon(f1_a, f1_b, alternative='two-sided')
        except ValueError:
            # All-zero differences after removing ties → no test
            p_val = 1.0

    sig  = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    bonf = 'YES' if p_val < bonferroni_alpha else '—'
    mean_a, mean_b = float(np.mean(f1_a)), float(np.mean(f1_b))
    winner = enc_a if mean_a > mean_b else enc_b
    delta  = mean_a - mean_b   # signed: +ve means A > B

    significance_results.append({
        'encoder_a': enc_a, 'encoder_b': enc_b,
        'p_value': p_val, 'significant': sig, 'bonferroni': bonf,
        'mean_a': mean_a, 'mean_b': mean_b,
        'winner': winner, 'delta_f1': delta
    })

    if p_val < 0.05:
        print(f"  {enc_a:>12s} vs {enc_b:>12s}  |  "
              f"{p_val:>10.5g}  {sig:>4s}  {bonf:>5s}  "
              f"{winner:>12s}  {delta:>+8.4f}")

sig_df = pd.DataFrame(significance_results)
print(f"\n  Total significant pairs (p<0.05, uncorrected): "
      f"{(sig_df['p_value'] < 0.05).sum()} / {len(sig_df)}")
print(f"  Total significant pairs (Bonferroni):           "
      f"{(sig_df['bonferroni'] == 'YES').sum()} / {len(sig_df)}")


In [ ]:
# =============================================================================
# CELL 5 — Significant wins summary
# =============================================================================
print("\n--- Significant Wins Summary (uncorrected p < 0.05) ---")
print(f"  {'Encoder':>12s} {'Type':>9s}   wins  losses   net")
print("-" * 50)

# Tally wins/losses for each encoder
rows = []
for enc in f1_encoders:
    wins = sig_df[
        (sig_df['winner'] == enc) & (sig_df['p_value'] < 0.05)
    ]
    losses = sig_df[
        (((sig_df['encoder_a'] == enc) & (sig_df['winner'] != enc)) |
         ((sig_df['encoder_b'] == enc) & (sig_df['winner'] != enc)))
        & (sig_df['p_value'] < 0.05)
    ]
    net = len(wins) - len(losses)
    rows.append({
        'encoder': enc, 'category': ENCODER_CATEGORY[enc],
        'wins': len(wins), 'losses': len(losses), 'net': net
    })

wins_df = pd.DataFrame(rows).sort_values('net', ascending=False).reset_index(drop=True)
for _, r in wins_df.iterrows():
    print(f"  {r['encoder']:>12s} {r['category']:>9s}    {r['wins']:>3d}    {r['losses']:>3d}   {r['net']:>+3d}")


## Section 3 — Synaptic Operations & Energy

**Source of truth.** Every value here comes from `mode1_test_results.csv`, which
nb4 populates by running each encoder's best checkpoint through the full test
loader inside `compute_efficiency_metrics()`. These are *measured* spike rates
and SynOps counts, not estimates from a hardcoded sparsity dict.

The Loihi-2 energy column (`energy_uj`) is computed as
`synops × 23.6 pJ × 10⁻⁶ µJ/pJ` per sample.

In [ ]:
# =============================================================================
# CELL 6 — Synaptic operations & energy (from measured test values)
# =============================================================================
print(f"\n{'=' * 80}")
print("SECTION 2: SYNAPTIC OPERATIONS & ENERGY EFFICIENCY")
print("=" * 80)

# ── SNN parameter count ──────────────────────────────────────────────────────
snn_params = ((NUM_INPUTS  * NUM_HIDDEN1 + NUM_HIDDEN1) +
              (NUM_HIDDEN1 * NUM_HIDDEN2 + NUM_HIDDEN2) +
              (NUM_HIDDEN2 * NUM_OUTPUTS + NUM_OUTPUTS) +
              3)  # 3 learnable beta scalars (one per LIF layer)
print(f"\n  SNN total parameters             : {snn_params:>12,d}")
print(f"  DASEventNet (ResNet-50) parameters : {RESNET50_PARAMS:>12,d}")
print(f"  Parameter reduction factor         : {RESNET50_PARAMS / snn_params:>12,.0f}x")

# ── Build efficiency table from measured test-set values ─────────────────────
print(f"\n  {'Encoder':>12s}  {'Cat':>4s}  {'InSpkRate':>10s}  {'L1SpkRate':>10s}  "
      f"{'MSynOps':>9s}  {'Energy(uJ)':>11s}  {'Test F1':>8s}  {'F1/MSynOps':>11s}")
print("-" * 100)

efficiency_data = []
for enc in ENCODER_ORDER:
    sub_test = test_df[test_df['encoder'] == enc]
    if len(sub_test) == 0:
        print(f"  {enc:>12s}  {ENCODER_CATEGORY[enc][:4]:>4s}  ── no test row, skipped ──")
        continue
    row = sub_test.iloc[0]

    in_rate  = float(row.get('input_spike_rate', np.nan))
    l1_rate  = float(row.get('l1_spike_rate',    np.nan))
    msynops  = float(row.get('msynops',          np.nan))
    synops   = float(row.get('synops',           msynops * 1e6 if not np.isnan(msynops) else np.nan))
    energy   = float(row.get('energy_uj',        np.nan))
    test_f1  = float(row.get('f1',               np.nan))
    inf_ms   = float(row.get('inference_time_ms', np.nan))

    cv_sub      = cv_df[cv_df['encoder'] == enc]
    cv_f1_mean  = float(cv_sub['f1'].mean()) if len(cv_sub) else np.nan
    cv_f1_std   = float(cv_sub['f1'].std())  if len(cv_sub) else np.nan

    f1_per_msynops = test_f1 / msynops if (msynops and not np.isnan(msynops) and msynops > 0) else np.nan

    efficiency_data.append({
        'encoder': enc, 'category': ENCODER_CATEGORY[enc],
        'input_spike_rate': in_rate, 'l1_spike_rate': l1_rate,
        'synops': synops, 'msynops': msynops, 'energy_uj': energy,
        'inference_time_ms': inf_ms,
        'test_f1': test_f1, 'cv_f1_mean': cv_f1_mean, 'cv_f1_std': cv_f1_std,
        'f1_per_msynops': f1_per_msynops,
    })

    print(f"  {enc:>12s}  {ENCODER_CATEGORY[enc][:4]:>4s}  "
          f"{in_rate:>10.4f}  {l1_rate:>10.4f}  "
          f"{msynops:>9.3f}  {energy:>11.4f}  "
          f"{test_f1:>8.4f}  {f1_per_msynops:>11.4f}")

# ── DASEventNet baseline row for comparison ──────────────────────────────────
resnet_energy_uj = RESNET50_FLOPS * RESNET50_ENERGY_PJ_PER_FLOP / 1e6  # pJ → µJ
print(f"\n  {'DASEventNet':>12s}  {'CNN':>4s}  {'dense':>10s}  {'dense':>10s}  "
      f"{RESNET50_FLOPS/1e6:>9.0f}  {resnet_energy_uj:>11.1f}  "
      f"{'~1.000*':>8s}  {1.0 / (RESNET50_FLOPS/1e6):>11.6f}")
print(f"\n  * DASEventNet F1≈1.000 reported by Yu et al. on 260 test samples (no CV).")

eff_df = pd.DataFrame(efficiency_data)

# ── Best per category ────────────────────────────────────────────────────────
print(f"\n  --- Best per category ---")
for cat in ['Baseline', 'Proposed']:
    cat_df = eff_df[eff_df['category'] == cat]
    if len(cat_df) == 0:
        continue
    best_f1   = cat_df.loc[cat_df['test_f1'].idxmax()]
    best_eff  = cat_df.loc[cat_df['f1_per_msynops'].idxmax()]
    print(f"  {cat:>10s}  Best F1: {best_f1['encoder']:>10s} ({best_f1['test_f1']:.4f})  |  "
          f"Best F1/MSynOps: {best_eff['encoder']:>10s} ({best_eff['f1_per_msynops']:.4f})")

if len(eff_df) > 0:
    best_overall_eff = eff_df.loc[eff_df['f1_per_msynops'].idxmax()]
    best_overall_f1  = eff_df.loc[eff_df['test_f1'].idxmax()]
    print(f"\n  Overall most efficient : {best_overall_eff['encoder']:>10s}  "
          f"(F1/MSynOps = {best_overall_eff['f1_per_msynops']:.4f})")
    print(f"  Overall best F1        : {best_overall_f1['encoder']:>10s}  "
          f"(F1 = {best_overall_f1['test_f1']:.4f})")


## Section 4 — Learned β Analysis

Each LIF layer in the SNN has one learnable β ∈ ℝ scalar (initial values
β₁=0.85, β₂=0.90, β₃=0.95 from nb4). β > 1 means the membrane potential
*grows* over time (leaky integrator with gain) rather than decaying — a sign
that the network is compensating for sparse input by accumulating evidence.

We parse `betas` from each CV row and report the mean ± std plus the fraction
of runs where β > 1.

In [ ]:
# =============================================================================
# CELL 7 — Learned beta analysis
# =============================================================================
print(f"\n{'=' * 80}")
print("SECTION 3: LEARNED β ANALYSIS")
print("=" * 80)

print(f"\n  {'Encoder':>12s}  {'b1 (init=0.85)':>18s}  {'b2 (init=0.90)':>18s}  "
      f"{'b3 (init=0.95)':>18s}  {'b2>1':>5s}  {'b3>1':>5s}")
print("-" * 90)

beta_data = []
for enc in active_encoders:
    sub = cv_df[cv_df['encoder'] == enc]
    b1 = sub['beta1'].dropna().values
    b2 = sub['beta2'].dropna().values
    b3 = sub['beta3'].dropna().values

    if len(b1) == 0 or len(b2) == 0 or len(b3) == 0:
        print(f"  {enc:>12s}  ── betas missing for this encoder ──")
        continue

    b2_gt1 = float(np.mean(b2 > 1.0)) * 100
    b3_gt1 = float(np.mean(b3 > 1.0)) * 100

    beta_data.append({
        'encoder': enc, 'category': ENCODER_CATEGORY[enc],
        'beta1_mean': float(np.mean(b1)), 'beta1_std': float(np.std(b1)),
        'beta2_mean': float(np.mean(b2)), 'beta2_std': float(np.std(b2)),
        'beta3_mean': float(np.mean(b3)), 'beta3_std': float(np.std(b3)),
        'beta2_gt1_pct': b2_gt1, 'beta3_gt1_pct': b3_gt1,
        'n_runs': len(b1),
    })

    print(f"  {enc:>12s}  {np.mean(b1):.4f}±{np.std(b1):.4f}    "
          f"{np.mean(b2):.4f}±{np.std(b2):.4f}    "
          f"{np.mean(b3):.4f}±{np.std(b3):.4f}    "
          f"{b2_gt1:>4.0f}%  {b3_gt1:>4.0f}%")

beta_df = pd.DataFrame(beta_data)

# ── Aggregate finding across all encoders ────────────────────────────────────
all_b2 = cv_df['beta2'].dropna().values
all_b3 = cv_df['beta3'].dropna().values
if len(all_b2) and len(all_b3):
    print(f"\n  KEY FINDING: β₂ > 1 in {np.mean(all_b2 > 1.0)*100:.0f}% "
          f"of all runs (n={len(all_b2)})")
    print(f"  KEY FINDING: β₃ > 1 in {np.mean(all_b3 > 1.0)*100:.0f}% "
          f"of all runs (n={len(all_b3)})")
    print(f"  → The SNN compensates for sparse input by learning to ACCUMULATE")
    print(f"    membrane potential rather than decay it (β > 1 = leaky integrator with gain).")


## Section 5 — Comparison with DASEventNet (published baseline)

> Yu, P., Zhu, T., Marone, C., Elsworth, D., & Yu, M. (2024).
> **DASEventNet: AI-Based Microseismic Detection on DAS Data from Utah FORGE.**
> *Journal of Geophysical Research: Solid Earth*, 129, e2024JB029102.

Direct numerical comparison is **not** strictly fair — different stimulation,
different label set (1,309 STA/LTA-derived events), different test-set size
(260 vs ours), no cross-validation. But the **architectural** comparison is
valid: ours is two orders of magnitude smaller and runs on event-driven
neuromorphic hardware.

In [ ]:
# =============================================================================
# CELL 8 — DASEventNet comparison table
# =============================================================================
print(f"\n{'=' * 80}")
print("SECTION 4: COMPARISON WITH DASEventNet (Yu et al., 2024)")
print("=" * 80)

# Pick best F1 from each category for the comparison
best_baseline_enc = (eff_df[eff_df['category'] == 'Baseline']
                     .sort_values('test_f1', ascending=False).iloc[0]['encoder'])
best_proposed_enc = (eff_df[eff_df['category'] == 'Proposed']
                     .sort_values('test_f1', ascending=False).iloc[0]['encoder'])

print(f"\n  Comparing: {best_baseline_enc} (best baseline)  |  "
      f"{best_proposed_enc} (best proposed)  |  DASEventNet (Yu et al. 2024)")
print()

bl_test = test_df[test_df['encoder'] == best_baseline_enc].iloc[0]
pr_test = test_df[test_df['encoder'] == best_proposed_enc].iloc[0]
bl_eff  = eff_df[eff_df['encoder']  == best_baseline_enc].iloc[0]
pr_eff  = eff_df[eff_df['encoder']  == best_proposed_enc].iloc[0]

print(f"  {'Metric':>22s}  {best_baseline_enc + ' (BL)':>22s}  "
      f"{best_proposed_enc + ' (PR)':>22s}  {'DASEventNet':>16s}")
print("-" * 95)

comparisons = [
    ('Architecture',     'FF-SNN (LIF)',                     'FF-SNN (LIF)',                     'ResNet-50 CNN'),
    ('Parameters',       f'{snn_params:,}',                  f'{snn_params:,}',                  f'{RESNET50_PARAMS:,}'),
    ('Encoding',         'Spike (baseline)',                 'Spike (proposed)',                 'Raw DAS amplitude'),
    ('Test F1',          f'{bl_test["f1"]:.4f}',             f'{pr_test["f1"]:.4f}',             '~1.000*'),
    ('Test Accuracy',    f'{bl_test["accuracy"]:.4f}',       f'{pr_test["accuracy"]:.4f}',       '1.000*'),
    ('Test AUC',         f'{bl_test.get("auc", float("nan")):.4f}',
                         f'{pr_test.get("auc", float("nan")):.4f}',
                         'not reported'),
    ('Test FNR',         f'{bl_test["fnr"]:.4f}',            f'{pr_test["fnr"]:.4f}',            '0.000*'),
    ('Test samples',     f'{N_TEST:,}',                       f'{N_TEST:,}',                       '260'),
    ('Cross-validation', f'{N_FOLDS}-fold × {N_SEEDS} seeds', f'{N_FOLDS}-fold × {N_SEEDS} seeds', 'single split'),
    ('Dataset',          'FORGE June 2024',                  'FORGE June 2024',                  'FORGE April 2022'),
    ('Hardware target',  'Neuromorphic (Loihi-2)',           'Neuromorphic (Loihi-2)',           'GPU'),
    ('MSynOps/inf',      f'{bl_eff["msynops"]:.3f}',         f'{pr_eff["msynops"]:.3f}',         f'{RESNET50_FLOPS/1e6:.0f} (FLOPs)'),
    ('Est. energy/inf',  f'{bl_eff["energy_uj"]:.4f} µJ',    f'{pr_eff["energy_uj"]:.4f} µJ',    f'{resnet_energy_uj:.1f} µJ'),
]

for label, v1, v2, v3 in comparisons:
    print(f"  {label:>22s}  {v1:>22s}  {v2:>22s}  {v3:>16s}")

print(f"\n  * DASEventNet achieves ≈100% F1 on only 260 test samples with no CV.")
print(f"    Our rigorous evaluation ({N_TEST:,} test, {N_FOLDS}-fold × {N_SEEDS}-seed CV) is more representative.")
print(f"    Architectural advantage of our approach: {RESNET50_PARAMS/snn_params:>5,.0f}× fewer parameters")
print(f"    and ≈{resnet_energy_uj/pr_eff['energy_uj']:>5,.0f}× lower energy on neuromorphic hardware (best proposed encoder).")


## Section 6 — Publication-Ready Summary Tables

Three tables for direct copy into the paper:

1. **Cross-validation results** (mean ± std over 15 runs) — primary metrics first.
2. **Held-out test set results** (n=600) — sorted by F1.
3. **Sweep-optimised encoder parameters** — confirmed values from the multi-file sweep
   (`2_2_FULL_Parameter_Sensitivity_Sweep__Multiple_data_.ipynb`).


In [ ]:
# =============================================================================
# CELL 9 — Table: Cross-Validation Results (mean ± std)
# =============================================================================
print(f"\n{'=' * 80}")
print("SECTION 5: PUBLICATION-READY TABLES")
print("=" * 80)

print(f"\n--- Table 1: Cross-Validation Results (mean ± std, "
      f"{N_FOLDS} folds × {N_SEEDS} seeds = {N_RUNS} runs) ---")
print(f"  Primary metrics: F1, Recall, FNR, FPR, AUPRC")
print()
print(f"  {'Encoder':>12s} {'Type':>9s}  {'F1':>13s}  {'Recall':>13s}  "
      f"{'FNR':>13s}  {'FPR':>13s}  {'AUPRC':>13s}")
print("-" * 100)

for enc in ENCODER_ORDER:
    sub = cv_df[cv_df['encoder'] == enc]
    if len(sub) == 0:
        continue
    cat = ENCODER_CATEGORY[enc]
    row = f"  {enc:>12s} {cat:>9s}"
    for m in PRIMARY_METRICS:
        if m in sub.columns:
            row += f"  {sub[m].mean():.3f}±{sub[m].std():.3f}"
        else:
            row += f"  {'N/A':>13s}"
    print(row)

print(f"\n  Secondary metrics: Accuracy, Balanced Acc, Precision, Specificity, AUC, MCC")
print()
print(f"  {'Encoder':>12s} {'Type':>9s}  {'Accuracy':>13s}  {'BalAcc':>13s}  "
      f"{'Precision':>13s}  {'Specificity':>13s}  {'AUC':>13s}  {'MCC':>13s}")
print("-" * 115)

for enc in ENCODER_ORDER:
    sub = cv_df[cv_df['encoder'] == enc]
    if len(sub) == 0:
        continue
    cat = ENCODER_CATEGORY[enc]
    row = f"  {enc:>12s} {cat:>9s}"
    for m in SECONDARY_METRICS:
        if m in sub.columns:
            row += f"  {sub[m].mean():.3f}±{sub[m].std():.3f}"
        else:
            row += f"  {'N/A':>13s}"
    print(row)


In [ ]:
# =============================================================================
# CELL 10 — Table: Held-out test set results (sorted by F1)
# =============================================================================
print(f"\n--- Table 2: Held-Out Test Set Results (n={N_TEST}, best checkpoint per encoder) ---")
print(f"  {'Rank':>4s}  {'Encoder':>12s} {'Type':>9s}  {'Acc':>7s}  {'Prec':>7s}  "
      f"{'Recall':>7s}  {'F1':>7s}  {'FNR':>7s}  {'AUC':>7s}  {'AUPRC':>7s}  {'MCC':>7s}")
print("-" * 110)

cols_for_table = ['accuracy', 'precision', 'recall', 'f1', 'fnr', 'auc', 'auprc', 'mcc']
test_sorted = test_df.sort_values('f1', ascending=False).reset_index(drop=True)
for i, row in test_sorted.iterrows():
    enc = row['encoder']
    if enc not in ENCODER_CATEGORY:
        continue
    cat = ENCODER_CATEGORY[enc]
    cells = ''
    for c in cols_for_table:
        v = row.get(c, np.nan)
        cells += f"  {v:>7.4f}" if (isinstance(v, (int, float)) and not pd.isna(v)) else f"  {'N/A':>7s}"
    print(f"  {i+1:>4d}  {enc:>12s} {cat:>9s}{cells}")


In [ ]:
# =============================================================================
# CELL 11 — Table: Sweep-Optimised Encoder Parameters
# =============================================================================
# Values confirmed by 2_2_FULL_Parameter_Sensitivity_Sweep__Multiple_data_.ipynb
# (400 event + 400 noise files, sparsity target 0.3-1.5%).
# These match the parameters used in 3_0_Preprocessing__2k_files_.ipynb.
print(f"\n--- Table 3: Sweep-Optimised Encoder Parameters ---")
print(f"  {'Encoder':>12s} {'Type':>9s}  {'Parameters':>55s}  "
      f"{'sp_ev%':>7s}  {'sp_no%':>7s}  {'sp_avg%':>8s}  {'SNR':>6s}")
print("-" * 110)

sweep_params = [
    # (encoder, params, sp_event%, sp_noise%, sp_avg%, snr)
    ('Rate',      'thr=0.85',                                          0.409, 0.514, 0.461, '3.09×'),
    ('Phase',     'thr=0.85',                                          0.444, 0.570, 0.507, '2.71×'),
    ('TTFS',      'thr=0.85  (converges to Phase on DAS)',             0.444, 0.570, 0.507, '2.71×'),
    ('Delta-Mod', 'thr=0.85, step_size=0.18',                          0.269, 0.339, 0.304, '2.63×'),
    ('PDE',       'kappa_phi=0.5, amp_kappa=3.0',                      0.672, 0.455, 0.563, '2.17×'),
    ('ATDE',      'alpha=3.0, tau_ms=165.0',                           0.416, 0.282, 0.349, '2.22×'),
    ('MASTE',     'lags=[1,3,8], alpha=2.5',                           0.366, 0.256, 0.311, '2.52×'),
    ('ST-MW',     'thr_t=0.18, thr_s=0.45, wt_ms=1.0, ws=8',           0.504, 0.609, 0.556, '5.40×'),
    ('AMSTE',     'alpha=3.0, lags=[1,3,8], min_votes=2, ws=16, thr_s=0.5',
                                                                       0.461, 0.272, 0.366, '3.91×'),
    ('SFBE',      'ratio_thr=10.0, sta_ms=8.0  (sp_ev/sp_no=6.4×)',    2.373, 0.370, 1.372, '0.83×'),
]

for enc, params, sp_ev, sp_no, sp_av, snr in sweep_params:
    cat = ENCODER_CATEGORY.get(enc, '?')
    print(f"  {enc:>12s} {cat:>9s}  {params:>55s}  "
          f"{sp_ev:>7.3f}  {sp_no:>7.3f}  {sp_av:>8.3f}  {snr:>6s}")


## Section 7 — Save Tables to CSV

All summary tables are saved alongside the raw nb4 results so the paper-writing
pipeline can pull them directly without re-running the notebook.

In [ ]:
# =============================================================================
# CELL 12 — Save all tables to CSV
# =============================================================================
saved_paths = []

# ── CV summary (mean ± std per encoder) ──────────────────────────────────────
cv_summary_rows = []
for enc in ENCODER_ORDER:
    sub = cv_df[cv_df['encoder'] == enc]
    if len(sub) == 0:
        continue
    row = {'encoder': enc, 'category': ENCODER_CATEGORY[enc], 'n_runs': len(sub)}
    for m in PRIMARY_METRICS + SECONDARY_METRICS:
        if m in sub.columns:
            row[f'{m}_mean'] = float(sub[m].mean())
            row[f'{m}_std']  = float(sub[m].std())
    cv_summary_rows.append(row)
cv_summary_df = pd.DataFrame(cv_summary_rows)
p = os.path.join(RESULTS_DIR, 'cv_summary_table.csv')
cv_summary_df.to_csv(p, index=False)
saved_paths.append(p)

# ── Efficiency table (test-set, measured) ────────────────────────────────────
p = os.path.join(RESULTS_DIR, 'efficiency_table.csv')
eff_df.to_csv(p, index=False)
saved_paths.append(p)

# ── Beta analysis ────────────────────────────────────────────────────────────
if len(beta_df) > 0:
    p = os.path.join(RESULTS_DIR, 'beta_analysis_table.csv')
    beta_df.to_csv(p, index=False)
    saved_paths.append(p)

# ── Statistical significance (all pairs + significant subset) ────────────────
if len(sig_df) > 0:
    p = os.path.join(RESULTS_DIR, 'statistical_significance.csv')
    sig_df.to_csv(p, index=False)
    saved_paths.append(p)

    sig_pairs = sig_df[sig_df['p_value'] < 0.05].sort_values('p_value')
    p = os.path.join(RESULTS_DIR, 'significant_pairs.csv')
    sig_pairs.to_csv(p, index=False)
    saved_paths.append(p)

# ── Wins/losses summary ──────────────────────────────────────────────────────
if len(wins_df) > 0:
    p = os.path.join(RESULTS_DIR, 'wins_losses_summary.csv')
    wins_df.to_csv(p, index=False)
    saved_paths.append(p)

print(f"\n{'=' * 80}")
print("FILES SAVED:")
print("=" * 80)
for p in saved_paths:
    print(f"  {p}")
print()
print("  cv_summary_table.csv         — CV mean±std (all metrics) for paper Table 1")
print("  efficiency_table.csv         — Measured test-set SynOps & energy for Table 2")
print("  beta_analysis_table.csv      — Learned β values per encoder")
print("  statistical_significance.csv — All pairwise Wilcoxon comparisons")
print("  significant_pairs.csv        — Subset with p < 0.05 (sorted by p)")
print("  wins_losses_summary.csv      — Per-encoder significant wins/losses")
print("=" * 80)


## Workflow Reminder

The 10-encoder benchmark pipeline:

1. **`1_0_Preprocessing__Single_files_.ipynb`** — sanity-check encoders on a single SEGY file.
2. **`2_2_FULL_Parameter_Sensitivity_Sweep__Multiple_data_.ipynb`** — confirm best params on 400 ev + 400 no SEGY files (sparsity target 0.3–1.5%).
3. **`3_0_Preprocessing__2k_files_.ipynb`** — batch-encode the full 2k+2k dataset with sweep-confirmed params → writes `Encoded_June_24/`.
4. **`4_0_Experiment_of_encoders_for_1D_SNN.ipynb`** — train 10 encoders × 5 folds × 3 seeds = 150 runs, then evaluate on the 15% held-out test set → writes `mode1_within_cv_results.csv` and `mode1_test_results.csv`.
5. **This notebook** — produce all paper-ready statistics, tables, and significance tests.

**Key paper framing:** systematic benchmark, not superiority claim — *"Rather than assuming domain-specific encoding is inherently superior, we test this hypothesis directly with paired statistical tests."*
